# Coronary artery flow: case setup and solve from a `.msh` + `case_setup.csv`

This notebook reads an existing Fluent mesh, applies the flow setup held in `case_setup.csv`, solves, and exports the results. Nothing about the physics is hard-coded in the notebook: every value comes from the CSV, and the CSV's `source` column ties each value back to `setup.xlsx` (Flow rows) or the setup audit.

| | Default in `case_setup.csv` | Source |
|---|---|---|
| Fluid | Blood, ρ = 1050 kg/m³, Carreau (μ₀ 0.056, μ∞ 0.00345 Pa·s, λ 3.313 s, n 0.3568) | setup.xlsx Flow |
| Domain | Quarter vessel (cut planes x = 0, y = 0, axis = +z), R0 = 1.38 mm, 25 % area stenosis | setup.xlsx Flow / Artery |
| Inlet | 0.5 m/s area-mean (peak systole), parabolic profile | setup.xlsx Flow; profile shape is a choice |
| Outlet / wall | 0 Pa gauge / no-slip | setup.xlsx Flow |
| Flow regime | `auto`: laminar below 0.50 area stenosis, Transition SST at or above it | setup.xlsx Flow |
| Numerics | Pressure-based steady, SIMPLEC, least-squares gradients, 2nd-order pressure, 2nd-order upwind, absolute residuals 1e-5 | audit §5.8 |

**How to use it**
1. Edit the run-parameter cell below (mesh file(s), CSV path, Fluent version).
2. Steps 1–2 need no Fluent: they read and check the CSV and the mesh file.
3. Step 4 reads the mesh, lists its zones and maps them to inlet / outlet / wall / symmetry with the `zone_*` patterns. Check the table before going on.
4. Step 5 runs the pre-solve gates (inlet area, inlet mass flow), then solves.

Requirements: `pip install ansys-fluent-core numpy pandas matplotlib`, plus Ansys Fluent 2024 R2 or later. The settings paths used here were checked against the PyFluent 0.42 definitions for 24.2, 25.1, 25.2, 26.1 and 27.1.

**Scope:** steady only. TAWSS and OSI need a transient run with a pulsatile inlet (the WIP sheet of setup.xlsx); that is the next iteration.

## 0. Run parameters
The only cell you normally edit. Physics lives in the CSV.

In [ ]:
MESH_FILES = [
    r"meshes\artery_quarter.msh",      # one or more Fluent meshes (.msh, .msh.h5, or a .cas/.cas.h5)
]
SETUP_CSV = "case_setup.csv"

FLUENT_VERSION = None      # e.g. "25.2"; None = newest installed
PROCS          = 4
SHOW_GUI       = False
OUT_DIR        = "artery_runs"

## Imports

In [ ]:
import datetime as _dt
import fnmatch
import json
import math
import platform
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Read and check the setup CSV
Each key has a parser. Unknown keys and missing required keys stop the notebook here, before Fluent starts.

In [ ]:
def _f(v):   return float(v)
def _i(v):   return int(float(v))
def _s(v):   return str(v).strip()
def _yes(v): return str(v).strip().lower() in ("yes", "y", "true", "1")
def _lst(v): return [p.strip() for p in str(v).split(";") if p.strip()]
def _vec(v):
    out = [float(p) for p in _lst(v)]
    if len(out) != 3:
        raise ValueError(f"expected 3 components separated by ';', got {v!r}")
    return out

def _one_of(*opts):
    def p(v):
        v = str(v).strip().lower()
        if v not in opts:
            raise ValueError(f"{v!r} is not one of {opts}")
        return v
    return p

# key -> (parser, required)
SPEC = {
    "density": (_f, True), "viscosity_model": (_one_of("carreau", "newtonian"), True),
    "carreau_mu0": (_f, True), "carreau_mu_inf": (_f, True),
    "carreau_lambda": (_f, True), "carreau_n": (_f, True),
    "carreau_shape_parameter": (_f, False),
    "mesh_units": (_one_of("m", "mm"), True), "R0": (_f, True),
    "sector_fraction": (_f, True), "axis_origin": (_vec, True), "axis_direction": (_vec, True),
    "stenosis_area": (_f, True),
    "zone_inlet": (_lst, True), "zone_outlet": (_lst, True), "zone_wall": (_lst, True),
    "zone_symmetry": (_lst, False),
    "inlet_mean_velocity": (_f, True), "inlet_profile": (_one_of("parabolic", "uniform"), True),
    "outlet_gauge_pressure": (_f, True),
    "viscous_model": (_one_of("auto", "laminar", "sst", "tsst"), True),
    "transition_threshold": (_f, True), "inlet_turb_intensity": (_f, True),
    "inlet_hydraulic_diameter": (_f, True),
    "time": (_one_of("steady", "transient"), True),
    "p_v_coupling": (_s, True), "gradient": (_s, True),
    "disc_pressure": (_s, True), "disc_convective": (_s, True),
    "residual_target": (_f, True), "max_iterations": (_i, True),
    "stability_tol": (_f, True), "stability_window": (_f, True), "mass_imbalance_tol": (_f, True),
    "inlet_area_tol": (_f, True), "inlet_mass_flow_tol": (_f, True),
    "export_field": (_yes, False),
}


def load_setup(path):
    raw = pd.read_csv(path, comment="#", dtype=str, keep_default_na=False)
    raw = raw[raw["key"].str.strip() != ""]
    raw["key"] = raw["key"].str.strip()
    dup = raw["key"][raw["key"].duplicated()].tolist()
    unknown = [k for k in raw["key"] if k not in SPEC]
    problems = []
    if dup:     problems.append(f"duplicate keys: {dup}")
    if unknown: problems.append(f"unknown keys: {unknown}")
    cfg = {}
    for k, v in zip(raw["key"], raw["value"]):
        if k not in SPEC:
            continue
        parser, _ = SPEC[k]
        if str(v).strip() == "":
            cfg[k] = [] if parser is _lst else None
            continue
        try:
            cfg[k] = parser(v)
        except Exception as e:
            problems.append(f"{k} = {v!r}: {e}")
    missing = [k for k, (_, req) in SPEC.items() if req and cfg.get(k) in (None, [])]
    if missing:
        problems.append(f"missing required keys: {missing}")
    if problems:
        raise ValueError("case_setup.csv problems:\n  " + "\n  ".join(problems))

    d = np.asarray(cfg["axis_direction"], float)
    if np.linalg.norm(d) == 0:
        raise ValueError("axis_direction is a zero vector")
    cfg["axis_direction"] = list(d / np.linalg.norm(d))
    cfg.setdefault("zone_symmetry", [])
    cfg.setdefault("export_field", False)
    cfg.setdefault("carreau_shape_parameter", None)
    if cfg["time"] != "steady":
        raise NotImplementedError("time = transient is not implemented in this version "
                                  "(pulsatile inlet, TAWSS/OSI sampling come next)")
    return cfg, raw


def resolve_model(cfg):
    if cfg["viscous_model"] != "auto":
        return cfg["viscous_model"]
    return "tsst" if cfg["stenosis_area"] >= cfg["transition_threshold"] else "laminar"


def flow_numbers(cfg):
    R0, U, rho = cfg["R0"], cfg["inlet_mean_velocity"], cfg["density"]
    A_nom = cfg["sector_fraction"] * math.pi * R0 ** 2
    D = 2 * R0
    return dict(
        A_inlet_nominal=A_nom,
        Q_model=U * A_nom, Q_full_vessel=U * math.pi * R0 ** 2,
        Re_mu_inf=rho * U * D / cfg["carreau_mu_inf"],
        Re_mu0=rho * U * D / cfg["carreau_mu0"] if cfg["viscosity_model"] == "carreau" else None,
        throat_radius=R0 * math.sqrt(1 - cfg["stenosis_area"]),
        U_throat_mean=U / (1 - cfg["stenosis_area"]),
    )


cfg, cfg_raw = load_setup(SETUP_CSV)
MODEL_KEY = resolve_model(cfg)
FLOW = flow_numbers(cfg)
print(f"viscous model -> {MODEL_KEY}  (viscous_model = {cfg['viscous_model']}, "
      f"stenosis {cfg['stenosis_area']:.2f} vs threshold {cfg['transition_threshold']:.2f})")
for k, v in FLOW.items():
    print(f"  {k:18s} {v:.6g}" if v is not None else f"  {k:18s} -")
cfg_raw[["key", "value", "units", "source"]]

## 2. Check the mesh files (no Fluent needed)
Fluent's `read_mesh` reads Fluent-format meshes only. A gmsh-native `.msh` has the same extension but starts with `$MeshFormat`, and Fluent will fail on it with an unhelpful message, so it is caught here.

In [ ]:
def inspect_mesh_file(p: Path) -> dict:
    if not p.exists():
        raise FileNotFoundError(p)
    name = p.name.lower()
    if name.endswith((".cas", ".cas.h5", ".cas.gz")):
        kind = "case"
    elif name.endswith((".msh", ".msh.h5", ".msh.gz")):
        kind = "mesh"
    else:
        raise ValueError(f"{p.name}: expected .msh / .msh.h5 / .cas / .cas.h5")
    if name.endswith(".msh"):
        head = p.open("rb").read(64)
        if head.lstrip().startswith(b"$MeshFormat"):
            raise ValueError(f"{p.name} is a gmsh-native mesh, not a Fluent mesh. "
                             "Write it from Fluent Meshing / ICEM / a Fluent-format writer, "
                             "or import it into Fluent (e.g. via CGNS) and save a .msh.h5.")
    return dict(file=str(p.resolve()), kind=kind, size_MB=round(p.stat().st_size / 1e6, 2))

mesh_info = [inspect_mesh_file(Path(m)) for m in MESH_FILES]
pd.DataFrame(mesh_info)

## 3. PyFluent helpers (version tolerant)
Settings paths that moved between releases are resolved at run time (e.g. `viscous` → `turbulence` in 27.1, `velocity` → `velocity_magnitude` in 25.1, discretisation under `spatial_discretization` from 25.1). `_choose` only sets a text option Fluent lists as allowed; otherwise it stops and prints Fluent's list.

In [ ]:
def _has(obj, name):
    try:
        return name in (list(getattr(obj, "child_names", [])) + list(getattr(obj, "command_names", [])))
    except Exception:
        return False

def _first(obj, *names):
    for n in names:
        if _has(obj, n):
            return getattr(obj, n)
    raise AttributeError(f"none of {names} under {getattr(obj, 'path', obj)}")

def _set(obj, value):
    if _has(obj, "value"):
        obj.value = value
    else:
        obj.set_state(value)

def _choose(setting, candidates, what="value"):
    try:
        allowed = list(setting.allowed_values())
    except Exception:
        allowed = None
    if allowed:
        for c in candidates:
            if c in allowed:
                setting.set_state(c)
                return c
        raise ValueError(f"{what}: none of {candidates} allowed; Fluent offers {allowed}")
    setting.set_state(candidates[0])
    return candidates[0]

def _names(named):
    try:
        return list(named.get_object_names())
    except Exception:
        try:
            return list(named.keys())
        except Exception:
            return []

def _create(named, name):
    if name in _names(named):
        return named[name]
    if _has(named, "create"):
        named.create(name=name)
    else:
        named[name] = {}
    return named[name]

def _cmd(cmd, **kwargs):
    known = list(getattr(cmd, "argument_names", []))
    alias = {"quantities": ["quantities", "cell_func_domain"]}
    call = {}
    for k, v in kwargs.items():
        for cand in alias.get(k, [k]):
            if cand in known:
                call[cand] = v
                break
    return cmd(**call)

def _set_locations(rep, attr_candidates, names):
    for a in attr_candidates:
        if _has(rep, a):
            try:
                setattr(rep, a, list(names))
                return a
            except Exception:
                continue
    raise AttributeError(f"could not set locations on {getattr(rep, 'path', rep)}")

def compute_reports(s, names):
    raw = s.solution.report_definitions.compute(report_defs=list(names))
    vals = {}
    for item in (raw or []):
        for k, v in item.items():
            vals[k] = float(v[0] if isinstance(v, (list, tuple)) else v)
    return vals

def read_monitor_file(path):
    if not path.exists():
        return None, None
    rows, header = [], None
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        if line.startswith("("):
            header = [h.strip('"') for h in line.strip("()").split('" "')]
            continue
        try:
            rows.append([float(v) for v in line.split()])
        except ValueError:
            continue
    return header, (np.array(rows) if rows else None)

def residual_history(solver):
    try:
        it, data = solver.monitors.get_monitor_set_data(monitor_set_name="residual")
        return {k: float(np.asarray(v)[-1]) for k, v in data.items()}, int(np.asarray(it)[-1])
    except Exception as e:
        print(f"[warn] residual history unavailable ({e})")
        return {}, None

### Zones
`list_zones` reads every face zone and its current type. `map_zones` matches them to roles with the `zone_*` patterns from the CSV, refuses a zone claimed by two roles, and then retypes zones whose type is wrong (for example an inlet written as `wall`).

In [ ]:
BC_CONTAINERS = {   # settings container -> Fluent zone type name
    "velocity_inlet": "velocity-inlet", "pressure_outlet": "pressure-outlet", "wall": "wall",
    "symmetry": "symmetry", "interior": "interior", "mass_flow_inlet": "mass-flow-inlet",
    "pressure_inlet": "pressure-inlet", "outflow": "outflow", "axis": "axis",
    "periodic": "periodic", "interface": "interface",
}
ROLE_TYPE = {"inlet": ("velocity_inlet", "velocity-inlet"),
             "outlet": ("pressure_outlet", "pressure-outlet"),
             "wall": ("wall", "wall"),
             "symmetry": ("symmetry", "symmetry")}


def list_zones(s):
    bc = s.setup.boundary_conditions
    zones = {}
    for cont, ztype in BC_CONTAINERS.items():
        if _has(bc, cont):
            for n in _names(getattr(bc, cont)):
                zones[n] = ztype
    return zones


def map_zones(s, cfg):
    zones = list_zones(s)
    roles = {r: [] for r in ROLE_TYPE}
    for role in roles:
        pats = [p.lower() for p in cfg.get(f"zone_{role}", [])]
        roles[role] = sorted(z for z in zones
                             if zones[z] != "interior" and any(fnmatch.fnmatch(z.lower(), p) for p in pats))
    claimed = {}
    for role, zs in roles.items():
        for z in zs:
            claimed.setdefault(z, []).append(role)
    clash = {z: r for z, r in claimed.items() if len(r) > 1}
    if clash:
        raise ValueError(f"zones matched by more than one role: {clash} - tighten the zone_* patterns")
    for role in ("inlet", "outlet", "wall"):
        if not roles[role]:
            raise ValueError(f"no zone matched zone_{role} = {cfg.get('zone_' + role)}; "
                             f"zones in the mesh: {sorted(zones)}")
    if cfg["sector_fraction"] < 1 and not roles["symmetry"]:
        raise ValueError("sector_fraction < 1 but no symmetry zone matched zone_symmetry")
    if cfg["inlet_profile"] == "parabolic" and len(roles["inlet"]) != 1:
        raise ValueError(f"parabolic inlet needs exactly one inlet zone, matched {roles['inlet']}")

    unassigned = [z for z, t in zones.items() if t != "interior" and z not in claimed]
    table = [dict(zone=z, type_in_mesh=t, role=(claimed.get(z, ["-"])[0] if t != "interior" else "interior"))
             for z, t in sorted(zones.items())]
    # retype
    bc = s.setup.boundary_conditions
    for role, zs in roles.items():
        _, new_type = ROLE_TYPE[role]
        wrong = [z for z in zs if zones[z] != new_type]
        if wrong:
            print(f"[zones] retyping {wrong} -> {new_type}")
            bc.set_zone_type(zone_list=wrong, new_type=new_type)
    if unassigned:
        print(f"[warn] boundary zones with no role, left as they are: "
              f"{[(z, zones[z]) for z in unassigned]}")
    interiors = [z for z, t in zones.items() if t == "interior"]
    return roles, interiors, pd.DataFrame(table)

## 4. Case setup functions
- `setup_general`: 3D pressure-based steady; mesh scaling if `mesh_units = mm`
- `setup_material`: blood copy of air, constant density, Carreau (or Newtonian μ∞); the Carreau group is printed back so you can compare it with the CSV
- `set_model`: laminar / SST k-ω / Transition SST
- `set_bcs`: inlet velocity (parabolic about the vessel axis, or uniform), 0 Pa outlet, turbulence inputs when needed
- `set_numerics`: SIMPLEC, least-squares gradients, 2nd-order schemes, absolute residuals
- `make_reports`: inlet area, mass flow in/out, inlet pressure (= Δp, outlet is 0 Pa), area-average and max WSS, max velocity; written to a monitor file every iteration

In [ ]:
MODELS = {  # key: (description, fluent viscous.model candidates, k_omega_model)
    "laminar": ("Laminar", ["laminar"], None),
    "sst":     ("SST k-omega", ["k-omega"], "sst"),
    "tsst":    ("Transition SST (gamma-Re_theta)", ["transition-sst", "transition-sst-model"], None),
}


def setup_general(s, cfg):
    if cfg["mesh_units"] == "mm":
        print("[mesh] scaling mm -> m")
        s.mesh.scale(x_scale=1e-3, y_scale=1e-3, z_scale=1e-3)
    gen = s.setup.general.solver
    _choose(gen.type, ["pressure-based"], "solver type")
    _choose(gen.time, ["steady"], "time")


def setup_material(s, cfg, fluid_zones):
    fl = s.setup.materials.fluid
    if "blood" not in _names(fl):
        fl.make_a_copy(from_="air", to="blood")
    mat = fl["blood"]
    _choose(mat.density.option, ["constant"], "density option")
    mat.density.value = cfg["density"]
    if cfg["viscosity_model"] == "newtonian":
        _choose(mat.viscosity.option, ["constant"], "viscosity option")
        mat.viscosity.value = cfg["carreau_mu_inf"]
    else:
        _choose(mat.viscosity.option, ["carreau", "non-newtonian-carreau"], "viscosity option")
        c = mat.viscosity.carreau
        c.time_constant = cfg["carreau_lambda"]
        c.power_law_index = cfg["carreau_n"]
        c.zero_shear_viscosity = cfg["carreau_mu0"]
        c.infinite_shear_viscosity = cfg["carreau_mu_inf"]
        if _has(c, "activation_energy"):          # no temperature dependence (energy is off)
            try:
                c.activation_energy = 0.0
            except Exception as e:
                print(f"[warn] could not zero the Carreau activation energy: {e}")
        if cfg.get("carreau_shape_parameter") is not None and _has(c, "shape_parameter"):
            c.shape_parameter = cfg["carreau_shape_parameter"]
        print("[material] Carreau group as Fluent holds it:")
        print(json.dumps(c.get_state(), indent=2, default=str))
    for z in fluid_zones:
        s.setup.cell_zone_conditions.fluid[z].general.material = "blood"

    rv = s.setup.reference_values
    rv.density = cfg["density"]
    rv.viscosity = cfg["carreau_mu_inf"]
    rv.length = 2 * cfg["R0"]
    rv.velocity = cfg["inlet_mean_velocity"]


def set_model(s, model_key):
    _, fl_models, kw = MODELS[model_key]
    visc = _first(s.setup.models, "viscous", "turbulence")
    chosen = _choose(visc.model, fl_models, "viscous model")
    if kw:
        _choose(visc.k_omega_model, [kw], "k-omega model")
    return chosen


def parabolic_expression(cfg, R_in):
    ox, oy, oz = cfg["axis_origin"]
    dx, dy, dz = cfg["axis_direction"]
    rx, ry, rz = f"(x-{ox:.10g}[m])", f"(y-{oy:.10g}[m])", f"(z-{oz:.10g}[m])"
    r2 = f"({rx}^2+{ry}^2+{rz}^2-({rx}*{dx:.10g}+{ry}*{dy:.10g}+{rz}*{dz:.10g})^2)"
    U = cfg["inlet_mean_velocity"]
    return f"max(0[m/s], 2*{U:.10g}[m/s]*(1-{r2}/({R_in:.10g}[m])^2))"


def set_bcs(s, cfg, roles, model_key, R_in):
    bcs = s.setup.boundary_conditions
    turb = model_key != "laminar"
    expr = None
    for z in roles["inlet"]:
        vi = bcs.velocity_inlet[z]
        _choose(vi.momentum.velocity_specification_method,
                ["Magnitude, Normal to Boundary"], "velocity spec")
        vmag = _first(vi.momentum, "velocity_magnitude", "velocity")
        if cfg["inlet_profile"] == "uniform":
            _set(vmag, cfg["inlet_mean_velocity"])
        else:
            expr = parabolic_expression(cfg, R_in)
            try:
                vmag.value = expr
            except Exception as e:
                print(f"[info] inline expression rejected ({e}); using a named expression")
                ne = _create(s.setup.named_expressions, "u_inlet_parabolic")
                ne.definition = expr
                vmag.value = "u_inlet_parabolic"
        if turb:
            t = vi.turbulence
            _choose(t.turbulence_specification, ["Intensity and Hydraulic Diameter"], "turb spec")
            t.turbulent_intensity = cfg["inlet_turb_intensity"]
            t.hydraulic_diameter = cfg["inlet_hydraulic_diameter"]
    for z in roles["outlet"]:
        po = bcs.pressure_outlet[z]
        _set(po.momentum.gauge_pressure, cfg["outlet_gauge_pressure"])
        if turb:
            t = po.turbulence
            _choose(t.turbulence_specification, ["Intensity and Hydraulic Diameter"], "backflow turb spec")
            t.backflow_turbulent_intensity = cfg["inlet_turb_intensity"]
            t.backflow_hydraulic_diameter = cfg["inlet_hydraulic_diameter"]
    return expr


def set_numerics(s, cfg):
    m = s.solution.methods
    _choose(m.p_v_coupling.flow_scheme, [cfg["p_v_coupling"]], "p-v coupling")
    sd = m.spatial_discretization if _has(m, "spatial_discretization") else m
    _choose(sd.gradient_scheme, [cfg["gradient"]], "gradient")
    ds = sd.discretization_scheme
    for eq in _names(ds):
        target = cfg["disc_pressure"] if eq == "pressure" else cfg["disc_convective"]
        try:
            _choose(ds[eq], [target], f"discretisation {eq}")
        except Exception as e:
            print(f"[warn] discretisation for '{eq}' left at default ({e})")
    res = s.solution.monitor.residual
    try:
        _choose(res.options.criterion_type, ["absolute"], "residual criterion")
    except Exception:
        pass
    for eq in _names(res.equations):
        res.equations[eq].check_convergence = True
        res.equations[eq].absolute_criteria = cfg["residual_target"]


def make_reports(s, roles, fluid_zones, out_dir, tag):
    rd = s.solution.report_definitions
    surf_loc = ["surface_names", "locations"]
    r = _create(rd.surface, "a_inlet")
    _choose(r.report_type, ["surface-area"], "area report")
    _set_locations(r, surf_loc, roles["inlet"])

    defs = []
    for nm, zs in (("mflow_inlet", roles["inlet"]), ("mflow_outlet", roles["outlet"])):
        r = _create(rd.flux, nm)
        _choose(r.report_type, ["flux-massflow"], "flux report")
        _set_locations(r, ["boundaries", "locations"], zs)
        defs.append(nm)
    for nm, rtype, fld, zs in (
            ("p_inlet_areaavg", ["surface-areaavg"], "pressure", roles["inlet"]),
            ("wss_wall_areaavg", ["surface-areaavg"], "wall-shear", roles["wall"]),
            ("wss_wall_max", ["surface-facetmax", "surface-vertexmax"], "wall-shear", roles["wall"])):
        r = _create(rd.surface, nm)
        _choose(r.report_type, rtype, nm)
        r.field = fld
        _set_locations(r, surf_loc, zs)
        defs.append(nm)
    r = _create(rd.volume, "vmax")
    _choose(r.report_type, ["volume-max"], "volume report")
    r.field = "velocity-magnitude"
    _set_locations(r, ["cell_zones", "locations"], fluid_zones)
    defs.append("vmax")

    rf = _create(s.solution.monitor.report_files, "monitors")
    rf.report_defs = defs
    rf.file_name = str((out_dir / f"{tag}_monitors.out").resolve())
    try:
        rf.frequency = 1
    except Exception:
        pass
    return defs


def export_results(s, cfg, roles, interiors, out_dir, tag, model_key):
    exp = s.file.export.ascii
    base = ["pressure", "x-velocity", "y-velocity", "z-velocity", "velocity-magnitude",
            "strain-rate-mag", "viscosity-lam"]
    turb = ["turb-kinetic-energy", "specific-diss-rate", "viscosity-turb"] if model_key != "laminar" else []
    wall_q = ["pressure", "wall-shear", "x-wall-shear", "y-wall-shear", "z-wall-shear",
              "strain-rate-mag", "viscosity-lam"]
    jobs = [("wall", roles["wall"], wall_q)]
    if roles["symmetry"]:
        jobs.append(("symmetry", roles["symmetry"], base + turb))
    if cfg["export_field"]:
        jobs.append(("field", interiors, base + turb))
    written = {}
    for label, surfs, qty in jobs:
        fn = (out_dir / f"{tag}_{label}.csv").resolve()
        for attempt in (qty, qty[:5]):
            try:
                _cmd(exp, file_name=str(fn), surface_name_list=list(surfs), delimiter="comma",
                     quantities=attempt, location="node")
                written[label] = str(fn)
                break
            except Exception as e:
                print(f"[warn] export {label} with {attempt} failed: {e}")
    return written

### Session, load and run functions
`run_case` stops before solving if either gate fails:
- **Inlet area** within `inlet_area_tol` of `sector_fraction · π · R0²`. This catches mm/m mix-ups and a full/quarter mismatch.
- **Inlet mass flow** after initialisation within `inlet_mass_flow_tol` of ρ·U·A. This catches a wrong axis origin/direction in the parabolic profile.

After the solve, a case is **CONVERGED** only if residuals are met, every monitor is flat over the last `stability_window` of iterations (≤ `stability_tol`), and the mass imbalance is ≤ `mass_imbalance_tol`.

In [ ]:
def launch_fluent_session(procs=PROCS, version=FLUENT_VERSION, gui=SHOW_GUI, cwd=OUT_DIR):
    import ansys.fluent.core as pyfluent
    Path(cwd).mkdir(parents=True, exist_ok=True)
    kw = dict(processor_count=procs, cwd=str(Path(cwd).resolve()))
    try:
        kw.update(precision=pyfluent.Precision.DOUBLE, dimension=pyfluent.Dimension.THREE,
                  mode=pyfluent.FluentMode.SOLVER,
                  ui_mode=pyfluent.UIMode.GUI if gui else pyfluent.UIMode.NO_GUI)
    except AttributeError:
        kw.update(precision="double", version="3d", mode="solver", show_gui=bool(gui))
    if version:
        kw["product_version"] = version
        for cand in (str(version), str(version) + ".0", str(version).replace(".", "")):
            try:
                kw["product_version"] = pyfluent.FluentVersion(cand)
                break
            except Exception:
                continue
    print(f"[fluent] launching: {kw}")
    solver = pyfluent.launch_fluent(**kw)
    return solver, getattr(solver, "settings", solver)


def load_mesh(s, info):
    print(f"[fluent] reading {info['file']}")
    if info["kind"] == "case":
        s.file.read_case(file_name=info["file"])
    else:
        s.file.read_mesh(file_name=info["file"])
    setup_general(s, cfg)
    try:
        s.mesh.check()
    except Exception as e:
        print(f"[warn] mesh check call failed: {e}")
    roles, interiors, table = map_zones(s, cfg)
    fluid_zones = _names(s.setup.cell_zone_conditions.fluid)
    if not fluid_zones:
        raise RuntimeError("no fluid cell zone in the mesh")
    return roles, interiors, fluid_zones, table


def run_case(solver, s, info, roles, interiors, fluid_zones, model_key=None):
    model_key = model_key or MODEL_KEY
    tag = f"{Path(info['file']).name.split('.')[0]}_{model_key}"
    cdir = out / tag
    cdir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print(f"\n==== {tag} ====")

    setup_material(s, cfg, fluid_zones)
    fl_model = set_model(s, model_key)
    defs = make_reports(s, roles, fluid_zones, cdir, tag)

    # gate 1: inlet area (a standard initialisation first, so reports can be evaluated)
    s.solution.initialization.standard_initialize()
    A_in = abs(compute_reports(s, ["a_inlet"])["a_inlet"])
    A_nom = FLOW["A_inlet_nominal"]
    ratio = A_in / A_nom
    print(f"[check] inlet area {A_in:.6e} m2 vs nominal {A_nom:.6e} m2 (ratio {ratio:.4f})")
    if abs(ratio - 1) > cfg["inlet_area_tol"]:
        f = cfg["sector_fraction"]
        hint = " - check R0, sector_fraction, mesh_units and zone_inlet"
        if 0.5e6 < ratio < 2e6:
            hint = " - the mesh looks like it is in mm: set mesh_units = mm"
        elif 0.5e-6 < ratio < 2e-6:
            hint = " - the mesh is being scaled but is already in m: set mesh_units = m"
        elif abs(ratio * f - 1) < 0.05:
            hint = " - the mesh looks like a full vessel: set sector_fraction = 1"
        raise RuntimeError(f"inlet area gate failed{hint}")
    R_in = math.sqrt(A_in / (cfg["sector_fraction"] * math.pi))

    expr = set_bcs(s, cfg, roles, model_key, R_in)
    set_numerics(s, cfg)
    s.solution.initialization.hybrid_initialize()

    # gate 2: inlet mass flow
    mdot_target = cfg["density"] * cfg["inlet_mean_velocity"] * A_in
    mdot0 = abs(compute_reports(s, ["mflow_inlet"])["mflow_inlet"])
    err = abs(mdot0 - mdot_target) / mdot_target
    print(f"[check] inlet mass flow {mdot0:.6e} kg/s vs rho*U*A {mdot_target:.6e} kg/s ({err:.3%})")
    if err > cfg["inlet_mass_flow_tol"]:
        raise RuntimeError("inlet mass-flow gate failed - check axis_origin / axis_direction "
                           "for the parabolic profile")

    max_it = cfg["max_iterations"]
    s.solution.run_calculation.iterate(iter_count=max_it)

    vals = compute_reports(s, defs)
    res_last, it_res = residual_history(solver)
    hdr, mon = read_monitor_file(cdir / f"{tag}_monitors.out")
    n_it = int(mon[-1, 0]) if mon is not None else it_res
    stab = {}
    if mon is not None and hdr is not None and len(mon) > 20:
        w = max(10, int(cfg["stability_window"] * len(mon)))
        tail = mon[-w:, 1:]
        for j, nm in enumerate(hdr[1:1 + tail.shape[1]]):
            col = tail[:, j]
            stab[nm] = float((col.max() - col.min()) / max(abs(col[-1]), 1e-30))
    mi, mo = vals.get("mflow_inlet", np.nan), vals.get("mflow_outlet", np.nan)
    imbalance = abs(mi + mo) / abs(mi) if mi else np.nan
    stopped_early = n_it is not None and n_it < max_it
    res_ok = stopped_early or (bool(res_last) and all(v <= cfg["residual_target"] for v in res_last.values()))
    stab_ok = bool(stab) and all(v <= cfg["stability_tol"] for v in stab.values())
    converged = res_ok and stab_ok and imbalance <= cfg["mass_imbalance_tol"]
    if converged:
        status = "CONVERGED"
    elif res_ok and not stab_ok:
        status = "RESIDUALS MET, MONITORS STILL DRIFTING"
    elif stab_ok:
        status = "MONITORS STEADY, RESIDUALS ABOVE TARGET"
    else:
        status = "NOT CONVERGED"
    print(f"[result] {tag}: {status}; iterations={n_it}; mass imbalance={imbalance:.2e}; "
          f"dp={vals.get('p_inlet_areaavg', np.nan):.2f} Pa; "
          f"WSS avg/max={vals.get('wss_wall_areaavg', np.nan):.3f}/{vals.get('wss_wall_max', np.nan):.3f} Pa")

    s.file.write_case_data(file_name=str((cdir / f"{tag}.cas.h5").resolve()))
    files = export_results(s, cfg, roles, interiors, cdir, tag, model_key)

    record = dict(tag=tag, mesh=info["file"], model=model_key, fluent_model=fl_model,
                  inlet_profile=cfg["inlet_profile"], inlet_expression=expr,
                  A_inlet=A_in, A_inlet_ratio=ratio, R_inlet=R_in,
                  mdot_target=mdot_target, mdot_init=mdot0, **FLOW,
                  iterations=n_it, final_residuals=res_last, residuals_met=res_ok,
                  monitor_relative_spread_last_window=stab, monitors_steady=stab_ok,
                  mass_flow_in=mi, mass_flow_out=mo, mass_imbalance_rel=imbalance,
                  dp_inlet_outlet=vals.get("p_inlet_areaavg"),
                  wss_wall_areaavg=vals.get("wss_wall_areaavg"), wss_wall_max=vals.get("wss_wall_max"),
                  vmax=vals.get("vmax"), status=status, converged=converged,
                  wall_time_s=round(time.time() - t0, 1), exports=files, zones=roles)
    with open(cdir / f"{tag}_summary.json", "w") as fh:
        json.dump(record, fh, indent=2, default=float)
    return record


SUMMARY_COLS = ["tag", "model", "iterations", "status", "mass_imbalance_rel", "dp_inlet_outlet",
                "wss_wall_areaavg", "wss_wall_max", "vmax", "A_inlet_ratio", "wall_time_s"]

## Step 3: launch Fluent (3D, double precision) and save the run config

In [ ]:
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)
shutil.copy(SETUP_CSV, out / "case_setup_used.csv")
with open(out / "run_config.json", "w") as fh:
    json.dump(dict(created=_dt.datetime.now().isoformat(timespec="seconds"),
                   python=platform.python_version(), mesh_files=mesh_info,
                   fluent_version=FLUENT_VERSION, procs=PROCS, model=MODEL_KEY,
                   setup=cfg, flow=FLOW), fh, indent=2, default=str)

solver, s = launch_fluent_session()
results = []

## Step 4: read the first mesh and map its zones
Check the table: every boundary zone should have the role you expect. If a pattern is wrong, fix `zone_*` in the CSV, re-run Step 1, then this cell.

In [ ]:
info = mesh_info[0]
roles, interiors, fluid_zones, zone_table = load_mesh(s, info)
print("fluid cell zones:", fluid_zones)
zone_table

## Step 5: solve the first mesh
Check the two `[check]` lines (inlet area, inlet mass flow) and then the status. Laminar Carreau at Re ≈ 420 (μ∞) should converge steadily.

In [ ]:
rec = run_case(solver, s, info, roles, interiors, fluid_zones)
results.append(rec)
pd.DataFrame(results)[SUMMARY_COLS].to_csv(out / "run_summary.csv", index=False)
{k: rec[k] for k in SUMMARY_COLS}

## Step 6: remaining meshes (if more than one)
Same setup on each further mesh file, e.g. a mesh-independence set. `run_summary.csv` is rewritten after each case.

In [ ]:
done = {r["mesh"] for r in results}
for info_i in mesh_info:
    if info_i["file"] in done:
        continue
    try:
        roles_i, interiors_i, fluid_i, _ = load_mesh(s, info_i)
        results.append(run_case(solver, s, info_i, roles_i, interiors_i, fluid_i))
    except Exception as e:
        print(f"[error] {info_i['file']}: {e}")
        results.append(dict(tag=Path(info_i["file"]).stem, mesh=info_i["file"], status=f"FAILED: {e}"))
    pd.DataFrame(results).reindex(columns=SUMMARY_COLS).to_csv(out / "run_summary.csv", index=False)
    done.add(info_i["file"])
pd.DataFrame(results).reindex(columns=SUMMARY_COLS)

## Step 7: checks on the solution
- **Monitor histories**: every monitor should be flat by the end.
- **Carreau check**: laminar viscosity against strain rate on the symmetry plane, overlaid on the Carreau law from the CSV. The points should sit on the curve; if they sit on a flat line the material did not take.
- **WSS along the vessel**: wall shear against axial position.

In [ ]:
def carreau(g, c):
    return c["carreau_mu_inf"] + (c["carreau_mu0"] - c["carreau_mu_inf"]) * \
        (1 + (c["carreau_lambda"] * g) ** 2) ** ((c["carreau_n"] - 1) / 2)

def read_export(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df

ax_o, ax_d = np.array(cfg["axis_origin"]), np.array(cfg["axis_direction"])
for r in results:
    if str(r.get("status", "")).startswith("FAILED"):
        continue
    cdir = out / r["tag"]
    hdr, mon = read_monitor_file(cdir / f"{r['tag']}_monitors.out")
    if mon is not None:
        names = hdr[1:mon.shape[1]]
        fig, axs = plt.subplots(1, len(names), figsize=(2.9 * len(names), 2.5))
        for ax, j, nm in zip(np.atleast_1d(axs), range(1, mon.shape[1]), names):
            ax.plot(mon[:, 0], mon[:, j], lw=1)
            ax.set_title(nm, fontsize=8); ax.set_xlabel("iteration", fontsize=8); ax.tick_params(labelsize=7)
        fig.suptitle(f"{r['tag']}: {r['status']}", fontsize=9); plt.tight_layout(); plt.show()

    fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
    ex = r.get("exports", {})
    if "symmetry" in ex and cfg["viscosity_model"] == "carreau":
        d = read_export(ex["symmetry"])
        g = d["strain-rate-mag"].to_numpy()
        gg = np.logspace(np.log10(max(g[g > 0].min(), 1e-3)), np.log10(g.max()), 200)
        axs[0].loglog(g, d["viscosity-lam"], ".", ms=2, alpha=0.4, label="Fluent (symmetry nodes)")
        axs[0].loglog(gg, carreau(gg, cfg), "k-", lw=1.2, label="Carreau, CSV constants")
        axs[0].set_xlabel("strain rate [1/s]"); axs[0].set_ylabel("viscosity [Pa s]"); axs[0].legend(fontsize=8)
    if "wall" in ex:
        d = read_export(ex["wall"])
        P = d[["x-coordinate", "y-coordinate", "z-coordinate"]].to_numpy()
        s_ax = (P - ax_o) @ ax_d
        axs[1].plot(s_ax * 1e3, d["wall-shear"], ".", ms=2, alpha=0.4)
        axs[1].set_xlabel("axial position [mm]"); axs[1].set_ylabel("WSS [Pa]")
    fig.suptitle(r["tag"], fontsize=9); plt.tight_layout(); plt.show()

## Step 8: close Fluent

In [ ]:
solver.exit()

## Output layout
```
artery_runs/
  case_setup_used.csv            exact CSV used for the run
  run_config.json                parsed setup, flow numbers, mesh files, Fluent version
  run_summary.csv                one row per mesh (status, dp, WSS avg/max, vmax)
  <mesh>_<model>/
      *.cas.h5 / *.dat.h5        Fluent case and data
      *_monitors.out             monitor history
      *_wall.csv                 wall nodes: pressure, WSS and components, viscosity
      *_symmetry.csv             symmetry-plane nodes: p, velocity, strain rate, viscosity
      *_field.csv                all interior nodes (only if export_field = yes)
      *_summary.json             full record
```